## *Directed Results*
# **Core Analyses of Vitamin D Signatures**

This notebook presents the **directed, hypothesis-driven analyses** of transcriptomic responses to Vitamin D and its analogs.  
Unlike exploratory analyses, here we focus on **predefined questions** (core signatures, dose–response, enrichment) using the modular utilities developed in `vitd_utils`.

All constants, parameters, and paths are centralized in `vitd_utils.config`, ensuring reproducibility and consistency across analyses.

## Section 1: Imports & Config.

In [ ]:
# Allow imports from src/vitd_utils
import sys
sys.path.append("../src")

# Core project utilities
from vitd_utils import config, idsymbols, coregenes, dose, gsea, plotting, stats, dataset

# Standard scientific libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display options
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

print("Results will be saved to:", config.RESULTS_DIR)
print("Figures will be saved to:", config.FIG_DIR, "| SAVE_FIGS =", config.SAVE_FIGS)

### 1.1. Load and validate

In [ ]:
EXP  = pd.read_parquet("../data/exports/expression_matrix_clean.parquet")
META = pd.read_csv("../data/exports/signature_metadata_clean.csv")

# Estandariza columnas (sig_id, cell_id, dose, analog)
META = dataset.standardize_meta(META)
print("Columns after standardize_meta:", META.columns.tolist())
print(META.filter(regex="dose", axis=1).head(2))  # ver 'dose' presente

# Alinea por sig_id
from vitd_utils import dataset as _ds
EXP, META = _ds.align_exp_meta(EXP, META)


## 2. Gene ID ↔ Symbol Mapping

Most LINCS L1000 resources use **gene IDs** as stable identifiers, while interpretation requires **gene symbols**.  
To ensure consistency, we build a robust mapping between IDs and symbols using `vitd_utils.idsymbols`.  
This step guarantees that downstream analyses (core genes, enrichment, plotting) always have readable gene names with safe fallbacks.


In [ ]:
# Load gene metadata (example: geneinfo_beta.txt already loaded in previous steps)
gene_info = pd.read_csv("../data/raw_data/geneinfo_beta.txt", sep="\t")

# Build ID → symbol mapping
sym_map = idsymbols.build_symbol_map(gene_info)

# Quick check
print("Mapping size:", sym_map.shape[0])
print("Examples:\n", sym_map.head())

# Test safe fallback (ID not in map should return itself)
print("Test mapping:", idsymbols.map_symbols_or_ids(["100", "102", "999"], sym_map)[:5])

## 3. Consensus Core Genes (definition & scoring)

**Goal.** Define a robust Vitamin D “core” signature (UP/DOWN genes) that recurs across contexts (here: cell lines), and compute a single **core score** per signature capturing the balance of UP vs DOWN core genes.

**Method.**
1) Build a gene × context matrix of effects (here: mean L1000 z-scores **per cell line**).
2) For each context, take the top/bottom `N_TOP` genes and **vote-count** across contexts.
3) Select `CORE_UP_N` / `CORE_DN_N` genes using a minimum vote threshold and deterministic tie-breakers (mean |effect|).
4) Compute **core_score** for every signature:  
   `core_score = mean(z(core_UP)) − mean(z(core_DN))` (column-wise centering).

All thresholds and sizes are centralized in `vitd_utils.config`.

In [ ]:
# Sanity alignment: keep only signatures present in both matrices
common_sig = [c for c in EXP.columns if c in set(META["sig_id"])]
EXP = EXP[common_sig].copy()
META = META.loc[META["sig_id"].isin(common_sig)].copy()

# --- Build gene × cell effects (mean across signatures within each cell line)
cell_indexer = META.set_index("sig_id")["cell_id"]
effects_by_cell = EXP.T.groupby(cell_indexer).mean().T

print("effects_by_cell shape:", effects_by_cell.shape)
display(effects_by_cell.iloc[:5, :5])

## 4. Dose–Response Analysis

**Goal.** Test whether Vitamin D analogs induce a monotonic transcriptomic response as dose increases, and quantify effect sizes (slopes).

**Method.**
1. Bin doses into "low" vs "high" categories for exploratory plots (`dose.binarize_dose`).
2. Test monotonicity with **Spearman correlation** (`dose.dose_monotonicity`).
3. Estimate slopes with **OLS regression** on log10(dose) (`dose.ols_hc3`) using HC3 robust errors.
4. Summarize slopes across cell lines and visualize with **forest plots** (`plotting.forest_from_models`).


### 4.1 Prepare dose metadata

In [ ]:
assert "dose" in META.columns, "Expected 'dose' in META after standardization."

META["log_dose"] = np.log10(META["dose"])
META["dose_bin"] = META.groupby("cell_id")["dose"].transform(lambda d: dose.binarize_dose(d).values)

META[["sig_id", "cell_id", "dose", "log_dose", "dose_bin"]].head()

In [ ]:
# 1) Build gene × cell effects
effects_by_cell = dataset.effects_by_cell(EXP, META)  # genes × cell_id
print("effects_by_cell:", effects_by_cell.shape)

# 2) Consensus core sets (UP/DOWN)
cons = coregenes.build_consensus_core(
    effects_by_cell,
    top_n=config.N_TOP,
    min_votes=config.VOTE_MIN,
    target_up=config.CORE_UP_N,
    target_dn=config.CORE_DN_N,
    min_non_na=10,
)
core_up_ids = cons["core_up"]
core_dn_ids = cons["core_dn"]
print(f"[core sets] UP={len(core_up_ids)} | DOWN={len(core_dn_ids)}")

# 3) Core score for every signature (columns of EXP)
core_scores = coregenes.core_score_for_matrix(
    effects=EXP,          # genes × sig_id
    core_up=core_up_ids,  # ID list (match EXP.index)
    core_dn=core_dn_ids,
    center=True,
)

# 4) Merge to META (standardized) by sig_id
if "core_score" in META.columns:
    META = META.drop(columns=["core_score"])
META = META.merge(core_scores.rename("core_score"),
                  left_on="sig_id", right_index=True, how="left")

# Sanity check
print("Has core_score?", "core_score" in META.columns, "| nulls:", META["core_score"].isna().sum())
display(META[["sig_id","cell_id","dose","core_score"]].head())


### 4.2 Monotonicity test (Spearman ρ)

In [ ]:
# Per-cell monotonicity of core_score vs dose
mono_results = (
    META.groupby("cell_id")
        .apply(lambda sub: dose.dose_monotonicity(sub["dose"], sub["core_score"]))
        .apply(pd.Series)
        .reset_index()
)

print(mono_results)

### 4.3 Forest plot — Dose–response slopes (HC3)

We estimate the slope of the dose–response (core_score ~ log10 dose) **per cell line** using OLS with **HC3 robust errors**.  
The forest plot shows the coefficient and its 95% confidence interval (CI); a vertical dashed line at 0 represents “no trend”.  
This complements the monotonicity test by quantifying **effect size** and uncertainty.

In [ ]:
# Ensure prerequisites are available
assert "dose" in META.columns, "Dose missing — run Section 0 standardization first."
assert "core_score" in META.columns, "core_score missing — run Section 3 consensus core scoring first."

# 1) Fit OLS-HC3 models per cell line
models, labels = [], []
for cell, sub in META.groupby("cell_id", dropna=False):
    sub = sub[["dose", "core_score"]].dropna()
    # Require at least 3 samples and 2 unique dose values
    if sub["dose"].nunique(dropna=True) < 2 or len(sub) < 3:
        continue
    try:
        m = dose.ols_hc3(sub["dose"].values, sub["core_score"].values)
        models.append(m)
        labels.append(str(cell))
    except Exception:
        # Skip groups where regression fails numerically
        continue

# 2) Summarize and plot
if models:
    # Summarize slopes with robust CI and p-values
    summary_df = dose.summarize_forest(models, labels)
    display(summary_df.sort_values("coef"))

    # Optionally export summary table
    if getattr(config, "SAVE_TABLES", False):
        out_csv = config.RESULTS_DIR / "dose_response_forest_by_cell.csv"
        summary_df.to_csv(out_csv, index=False)
        print(f"[saved] {out_csv}")

    # Forest plot
    ax = plotting.forest_from_models(
        models, labels,
        title="Dose–response slope (core_score ~ log10 dose) by cell line",
        sort="coef"
    )
    plotting.savefig(filename="forest_dose_response_by_cell.png")
    plt.show()
else:
    print("[info] No groups had enough dose variation to fit OLS-HC3.")


### Interpretation — Dose–response slopes (HC3)

Across cell lines, OLS–HC3 slopes for *core_score ~ log10(dose)* are positive and statistically significant in most contexts, indicating a dose-dependent induction of the Vitamin D core response:

- **MCF7**: largest slope, narrow CI, *p* ≪ 1e-6 → strong dose dependence.
- **A549** and **PC3**: clearly positive slopes with tight CIs (***p* < 1e-5**), consistent dose dependence.
- **U2OS**: positive slope with wider CI; still significant (*p* ≈ 0.043), suggesting a weaker but present trend.
- **HA1E**: small slope, CI overlaps zero (*p* ≈ 0.15), indicating limited or context-specific dose dependence.

Overall, these results support a **monotonic, dose-responsive activation** of the Vitamin D core signature in most cell lines, with effect sizes varying by context.

---

## 5.1 Groupwise Spearman correlations (with FDR)

**Goal.** Quantify monotonic dose–response trends by computing **Spearman’s ρ** between `log10(dose)` and `core_score` **within each cell line**.  
**Multiple-testing control.** We report Benjamini–Hochberg **FDR** across cell lines to control the expected false discovery rate.

**Why Spearman?** It is rank-based and robust to non-linearity and mild outliers, complementing OLS–HC3 slope estimates.


In [ ]:
# Ensure required columns exist
assert {"cell_id", "log_dose", "core_score"}.issubset(META.columns), \
    "Missing required columns. Run Sections 3 and 4.1 first."

# Prepare a minimal, clean table
df = META[["cell_id", "log_dose", "core_score"]].dropna().copy()

# Compute Spearman by group (prefer utils; fall back to a safe inline implementation if needed)
try:
    # Expected signature: stats.spearman_by_group(df, group_col, x_col, y_col)
    spearman_df = stats.spearman_by_group(df, group_col="cell_id",
                                          x_col="log_dose", y_col="core_score")
except Exception:
    # Safe fallback: pure pandas/scipy implementation
    from scipy.stats import spearmanr

    rows = []
    for cell, sub in df.groupby("cell_id", dropna=False):
        # Require at least 3 samples and 2 unique x values
        if sub["log_dose"].nunique() < 2 or len(sub) < 3:
            rows.append({"cell_id": str(cell), "rho": np.nan, "pvalue": np.nan, "n": len(sub)})
            continue
        r, p = spearmanr(sub["log_dose"], sub["core_score"], nan_policy="omit")
        rows.append({"cell_id": str(cell), "rho": r, "pvalue": p, "n": len(sub)})
    spearman_df = pd.DataFrame(rows)

# Add BH–FDR (prefer utils; fall back to statsmodels if needed)
try:
    # Expected signature: stats.add_fdr(df, p_col, new_col="fdr", method="bh")
    spearman_df = stats.add_fdr(spearman_df, p_col="pvalue", new_col="fdr", method="bh")
except Exception:
    from statsmodels.stats.multitest import multipletests
    p = spearman_df["pvalue"].values
    mask = ~np.isnan(p)
    fdr = np.full_like(p, np.nan, dtype=float)
    if mask.sum() > 0:
        _, q, _, _ = multipletests(p[mask], alpha=0.05, method="fdr_bh")
        fdr[mask] = q
    spearman_df["fdr"] = fdr

# Nicely formatted table
spearman_df = (
    spearman_df.rename(columns={"cell_id": "label", "rho": "spearman_rho"})
               .sort_values(["fdr", "spearman_rho"], ascending=[True, False])
               .reset_index(drop=True)
)

display(spearman_df)

# Optional: save results
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "spearman_by_cell_fdr.csv"
    spearman_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")


## 5.2 Bootstrap CIs for Spearman’s ρ

**Goal.** Quantify uncertainty in the groupwise Spearman correlations using nonparametric **bootstrap** CIs.  
This complements p-values/FDR with effect-size intervals that are robust to non-normality.


In [ ]:
def _spearman_stat(xy_df: pd.DataFrame) -> float:
    # small helper that returns Spearman's rho
    from scipy.stats import spearmanr
    r, _ = spearmanr(xy_df["log_dose"], xy_df["core_score"], nan_policy="omit")
    return r

rows = []
for cell, sub in df.groupby("cell_id", dropna=False):
    sub = sub.copy()
    n = len(sub)
    if sub["log_dose"].nunique() < 2 or n < 3:
        rows.append({"label": str(cell), "rho": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": n})
        continue
    # Prefer utils if available; otherwise fallback to a simple bootstrap
    try:
        # Expected signature: stats.bootstrap_ci(data, stat_fn, n_boot=2000, ci=0.95, seed=0)
        ci_low, ci_high, point = stats.bootstrap_ci(sub, _spearman_stat, n_boot=4000, ci=0.95, seed=0)
        rows.append({"label": str(cell), "rho": point, "ci_low": ci_low, "ci_high": ci_high, "n": n})
    except Exception:
        # Basic bootstrap fallback
        rng = np.random.default_rng(0)
        boots = []
        for _ in range(4000):
            idx = rng.integers(0, n, n)  # sample with replacement
            boots.append(_spearman_stat(sub.iloc[idx]))
        boots = np.array(boots)
        ci_low, ci_high = np.nanpercentile(boots, [2.5, 97.5])
        rows.append({"label": str(cell), "rho": _spearman_stat(sub), "ci_low": ci_low, "ci_high": ci_high, "n": n})

boot_df = pd.DataFrame(rows).sort_values("rho", ascending=False).reset_index(drop=True)
display(boot_df)

# Optional: save table
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "spearman_by_cell_bootstrap_ci.csv"
    boot_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

# Point-interval plot
plt.figure(figsize=(6, 3.6))
y = np.arange(len(boot_df))[::-1]
plt.hlines(y, boot_df["ci_low"], boot_df["ci_high"])
plt.plot(boot_df["rho"], y, "o")
plt.axvline(0, linestyle="--", linewidth=1)
plt.yticks(y, boot_df["label"])
plt.xlabel("Spearman ρ (with 95% CI)")
plt.title("Dose–response monotonicity by cell line")
plt.tight_layout()
plt.show()

### Interpretation — Dose–response monotonicity

MCF7, A549, and PC3 show strong and consistent monotonic dose–response patterns, with Spearman’s ρ around 0.55–0.63 and narrow CIs excluding zero.  
U2OS displays a weaker but still positive trend with wider uncertainty, while HA1E shows no clear monotonicity, with a CI overlapping zero.  
Overall, most cell lines support a robust dose-dependent activation of the Vitamin D core signature.

## 5.3 Quick OLS slopes (polyfit)

**Goal.** Provide a simple, effect-size–oriented summary of dose–response strength in each cell line.  
We fit a straight line `core_score ~ log10(dose)` using numpy’s `polyfit`, which is fast and stable but does not provide robust errors.  
These slopes complement the HC3 regression (Section 4.3) by offering an easy-to-interpret Δy/Δx metric.


In [ ]:
# Ensure required columns are available
assert {"cell_id", "log_dose", "core_score"}.issubset(META.columns)

rows = []
for cell, sub in META.groupby("cell_id", dropna=False):
    slope = stats.fit_slope_ols(sub, x="log_dose", y="core_score")
    rows.append({"label": str(cell), "slope": slope, "n": len(sub)})

slopes_df = pd.DataFrame(rows).sort_values("slope", ascending=False).reset_index(drop=True)
display(slopes_df)

# Optional: save results
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "quick_slopes_by_cell.csv"
    slopes_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

# Simple bar plot of slopes
plt.figure(figsize=(6, 3.6))
plt.barh(slopes_df["label"], slopes_df["slope"], color="steelblue")
plt.axvline(0, linestyle="--", linewidth=1, color="black")
plt.xlabel("OLS slope (Δ core_score per Δ log10 dose)")
plt.title("Quick dose–response slopes by cell line")
plt.tight_layout()
plt.show()


### Interpretation — Quick slopes

MCF7 and A549 exhibit the steepest slopes (~0.45–0.50), indicating strong dose-dependent increases in the Vitamin D core response.  
PC3 and U2OS show moderate slopes (~0.29–0.30), consistent with weaker but still positive trends.  
HA1E displays only a minor slope (~0.13), suggesting little or no consistent dose dependence in this context.  
Overall, effect sizes align with previous analyses (Spearman and HC3 regression), reinforcing the robustness of dose–response induction across most cell lines.

## 5.4 Bootstrap CIs for OLS slopes

**Goal.** Quantify the uncertainty of quick OLS slope estimates (`core_score ~ log10 dose`) using nonparametric bootstrap confidence intervals.  
This provides robust effect-size intervals that do not rely on parametric assumptions, complementing the HC3 regression (Section 4.3).


In [ ]:
rows = []
for cell, sub in META.groupby("cell_id", dropna=False):
    sub = sub[["log_dose", "core_score"]].dropna()
    n = len(sub)
    if sub["log_dose"].nunique() < 2 or n < 3:
        rows.append({"label": str(cell), "slope": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": n})
        continue
    # Bootstrap CI using the utility if available
    try:
        lo, hi = stats.bootstrap_ci(stats.fit_slope_ols(sub), B=4000, alpha=0.05)
        slope_val = stats.fit_slope_ols(sub)
    except Exception:
        # Manual fallback
        rng = np.random.default_rng(0)
        boots = []
        x, y = sub["log_dose"].values, sub["core_score"].values
        for _ in range(4000):
            idx = rng.integers(0, n, n)
            slope = stats.fit_slope_ols(pd.DataFrame({"log_dose": x[idx], "core_score": y[idx]}))
            boots.append(slope)
        boots = np.array([b for b in boots if not np.isnan(b)])
        slope_val = stats.fit_slope_ols(sub)
        lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    rows.append({"label": str(cell), "slope": slope_val, "ci_low": lo, "ci_high": hi, "n": n})

boot_slopes_df = pd.DataFrame(rows).sort_values("slope", ascending=False).reset_index(drop=True)
display(boot_slopes_df)

# Optional: save
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "quick_slopes_bootstrap_ci.csv"
    boot_slopes_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

# Plot
plt.figure(figsize=(6, 3.6))
y = np.arange(len(boot_slopes_df))[::-1]
plt.hlines(y, boot_slopes_df["ci_low"], boot_slopes_df["ci_high"])
plt.plot(boot_slopes_df["slope"], y, "o")
plt.axvline(0, linestyle="--", linewidth=1)
plt.yticks(y, boot_slopes_df["label"])
plt.xlabel("OLS slope (Δ core_score per Δ log10 dose) with 95% CI")
plt.title("Bootstrap CIs for quick dose–response slopes")
plt.tight_layout()
plt.show()


### Interpretation — Bootstrap CIs for slopes

MCF7 and A549 show the strongest dose–response effects, with slopes ~0.45–0.50 and CIs well above zero.  
PC3 and U2OS also display positive slopes, though with wider intervals, indicating moderate but consistent trends.  
HA1E’s slope is small (~0.13) with a CI overlapping zero, suggesting no reliable dose dependence in this context.  
Together, the bootstrap intervals reinforce robust dose–dependent activation of the Vitamin D core signature in most cell lines, especially in MCF7 and A549.
